In [ ]:
!pip install -q chromadb sentence-transformers groq pypdf

In [ ]:
!pip install pypdf sentence-transformers chromadb groq

In [13]:
from google.colab import files

uploaded = files.upload()

Saving archana_resume.pdf to archana_resume (1).pdf
Saving Resume - Sriram K .pdf to Resume - Sriram K .pdf
Saving HARINI RESUME.pdf to HARINI RESUME (1).pdf


In [14]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

In [16]:
documents=[]
ids=[]
for filename in uploaded.keys():
  pdf=PdfReader(filename)
  text=""
  for page in pdf.pages:
    page_text = page.extract_text()
    if page_text:
      text += page_text

  documents.append(text)
  ids.append(filename)
print("uploaded resumes:")
print(ids)

uploaded resumes:
['archana_resume (1).pdf', 'Resume - Sriram K .pdf', 'HARINI RESUME (1).pdf']


In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents).tolist()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
client_db = chromadb.Client()
collection = client_db.get_or_create_collection(
    name = "resume_collection"
)
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

In [19]:
query = "Python Developer"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)
print("\nTop Matching Resumes:\n")

for i, file in enumerate(results["ids"][0]):
    print(f"Rank {i+1}: {file}")


Top Matching Resumes:

Rank 1: HARINI RESUME (1).pdf
Rank 2: archana_resume (1).pdf
Rank 3: Resume - Sriram K .pdf


In [20]:
context = "\n\n".join(results["documents"][0])

In [21]:
client = Groq(
    api_key="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"

)

prompt = f"""
Job Role: Python Developer

Resumes:
{context}

Analyze the resumes and provide:

1. Ranking of candidates
2. Best candidate
3. Matching skills
4. Short reason
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print("\nAI Analysis:\n")
print(response.choices[0].message.content)


AI Analysis:

Based on the provided resumes, here's the analysis:

**Ranking of Candidates:**
1. S Harini
2. Sri Ram K

**Best Candidate:**
S Harini

**Matching Skills:**
- Python
- Machine Learning
- Data Science
- Web development (HTML, CSS, JavaScript)
- Streamlit
- Scikit-learn

**Short Reason:**
S Harini is the best candidate due to her strong foundation in machine learning, data science, and web development. Her experience in building AI-powered projects, such as the job recommendation system and farmer advisory system, demonstrates her ability to apply theoretical concepts to real-world problems. Additionally, her certification in machine learning using Python and generative AI showcases her commitment to staying up-to-date with industry trends. Overall, S Harini's technical skills, project experience, and certifications make her a well-rounded candidate for a Python developer role.
